# Explore ICUdata DuckDB

Read-only tour of the combined ICUdata database (built by `duckdb_database_creator`).

**Schema layers in one file:**
- `omop.*` — the main interface: combined, public-facing OMOP views (the `hospital` column is dropped) plus generated tables `dictionary`, `source_to_concept_map`, `cdm_source`.
- `cze.*`, `vumc.*`, … — per-hospital views of the same OMOP tables (use these when you need a single site).
- `icudata.*` — internal bookkeeping: `version` / `changelog` only (no `omop` copy exists).

OMOP table names are singular: `device_exposure` (not `devices`), `measurement` (not `measurements`).

Run this on MyDRE, where the DB lives.

In [ ]:
import duckdb

DB = "/mnt/data/icudata/ICUdata_0_4_0.duckdb"  # the new build
con = duckdb.connect(DB, read_only=True)

## 1. All tables (every schema)

In [ ]:
con.sql("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
ORDER BY table_schema, table_name
""")

## 2. Person — `omop.person`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.person"))
con.sql("SELECT * FROM omop.person LIMIT 100")

## 3. Visits — `omop.visit_occurrence`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.visit_occurrence"))
con.sql("SELECT * FROM omop.visit_occurrence LIMIT 100")

## 4. Death — `omop.death`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.death"))
con.sql("SELECT * FROM omop.death LIMIT 100")

## 5. Measurements — `omop.measurement`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.measurement"))
con.sql("SELECT * FROM omop.measurement LIMIT 100")

## 6. Devices — `omop.device_exposure`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.device_exposure"))
con.sql("SELECT * FROM omop.device_exposure LIMIT 100")

## 7. Measurements for CZE — `cze.measurement`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM cze.measurement"))
con.sql("SELECT * FROM cze.measurement LIMIT 100")

## 8. Version history & changelog

These two are bookkeeping tables and live **only** in `icudata` — there is no `omop` copy.

In [ ]:
display(con.sql("SELECT version, created_at, hospitals, tables FROM icudata.version ORDER BY version"))
con.sql("""
SELECT id, from_version, to_version, change_type, entity, details
FROM icudata.changelog
ORDER BY id DESC
LIMIT 200
""")

## 9. Data dictionary — `omop.dictionary`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM omop.dictionary"))
con.sql("SELECT * FROM omop.dictionary LIMIT 200")

## 10. Source-to-concept map — `omop.source_to_concept_map`

In [ ]:
display(con.sql("SELECT COUNT(*) AS n_mappings FROM omop.source_to_concept_map"))
con.sql("SELECT * FROM omop.source_to_concept_map LIMIT 200")

## 11. Quick overviews

In [ ]:
# Build metadata: version, vocabulary version, release date
con.sql("SELECT * FROM omop.cdm_source")

In [ ]:
# Row counts for every combined OMOP table
tbls = [r[0] for r in con.sql("SELECT table_name FROM information_schema.tables WHERE table_schema='omop' ORDER BY table_name").fetchall()]
union = " UNION ALL ".join(f'SELECT \'{t}\' AS table_name, COUNT(*) AS n_rows FROM omop."{t}"' for t in tbls)
con.sql(union + " ORDER BY n_rows DESC")

In [ ]:
# Per-hospital row counts. omop.* drops the hospital column, so this reads the
# per-hospital public schemas (cze, vumc, ...) discovered from the catalog.
hosp = [r[0] for r in con.sql("SELECT table_schema FROM information_schema.tables WHERE table_name='measurement' AND table_schema NOT IN ('omop','icudata') ORDER BY table_schema").fetchall()]
union = " UNION ALL ".join(f"SELECT '{h}' AS hospital, COUNT(*) AS n_measurements FROM {h}.measurement" for h in hosp)
con.sql(union + " ORDER BY n_measurements DESC")

In [ ]:
# Top measured concepts, enriched with concept name + vocabulary
con.sql("""
SELECT m.measurement_concept_id,
       c.concept_name,
       c.vocabulary_id,
       COUNT(*) AS n
FROM omop.measurement m
LEFT JOIN omop.concept c ON m.measurement_concept_id = c.concept_id
GROUP BY 1, 2, 3
ORDER BY n DESC
LIMIT 25
""")

In [ ]:
# Data coverage: time span and distinct patients
con.sql("""
SELECT MIN(measurement_datetime) AS first_measurement,
       MAX(measurement_datetime) AS last_measurement,
       COUNT(DISTINCT person_id) AS n_patients
FROM omop.measurement
""")

## 12. Practical OMOP analyses

The concept IDs below were picked from the ICUdata `dictionary.csv` (highest patient coverage). Each use case joins a *fact* table to `person` / `visit_occurrence` / `death` / the `concept` vocabulary — the bread-and-butter of OMOP querying.

| Default | concept_id | table |
|---|---|---|
| Invasive mean BP (MAP) | 21490852 | measurement |
| SpO₂ (pulse oximetry) | 40762499 | measurement |
| Endotracheal tube | 2000000801 | device_exposure |

Edit the parameters cell to analyse other concepts.

In [ ]:
# ---- Parameters: concept_ids sourced from the ICUdata dictionary.csv ----
VITAL_CONCEPT_IDS = [21490852, 40762499]   # Invasive mean BP (MAP), SpO2 (pulse oximetry)
VENT_DEVICE_CONCEPT_IDS = [2000000801]      # Endotracheal tube

csv = lambda xs: ", ".join(str(x) for x in xs)  # helper: format an id list for SQL IN (...)

### Use case 1 — Cohort & first-24 h vitals

In [ ]:
# Cohort demographics: person + visit_occurrence + concept (sex, age)
con.sql("""
SELECT COALESCE(gc.concept_name, 'Unknown') AS sex,
       COUNT(DISTINCT p.person_id) AS n_patients,
       COUNT(*) AS n_visits,
       round(median(date_diff('year', p.birth_datetime, v.visit_start_datetime)), 1) AS median_age
FROM omop.person p
JOIN omop.visit_occurrence v ON p.person_id = v.person_id
LEFT JOIN omop.concept gc ON p.gender_concept_id = gc.concept_id
GROUP BY 1
ORDER BY n_patients DESC
""")

In [ ]:
# Vital values within the first 24 h: measurement + visit_occurrence + concept
con.sql(f"""
SELECT m.person_id,
       mc.concept_name AS vital,
       m.value_as_number,
       m.unit_source_value AS unit,
       date_diff('hour', v.visit_start_datetime, m.measurement_datetime) AS hours_into_stay
FROM omop.measurement m
JOIN omop.visit_occurrence v ON m.visit_occurrence_id = v.visit_occurrence_id
LEFT JOIN omop.concept mc ON m.measurement_concept_id = mc.concept_id
WHERE m.measurement_concept_id IN ({csv(VITAL_CONCEPT_IDS)})
  AND m.value_as_number IS NOT NULL
  AND m.measurement_datetime >= v.visit_start_datetime
  AND m.measurement_datetime <  v.visit_start_datetime + INTERVAL 24 HOUR
ORDER BY m.person_id, hours_into_stay
LIMIT 200
""")

In [ ]:
# Vital summary by sex: measurement + person + concept
con.sql(f"""
SELECT mc.concept_name AS vital,
       COALESCE(gc.concept_name, 'Unknown') AS sex,
       COUNT(*) AS n_values,
       COUNT(DISTINCT m.person_id) AS n_patients,
       round(quantile_cont(m.value_as_number, 0.25), 2) AS p25,
       round(median(m.value_as_number), 2) AS p50,
       round(quantile_cont(m.value_as_number, 0.75), 2) AS p75,
       any_value(m.unit_source_value) AS unit
FROM omop.measurement m
JOIN omop.person p ON m.person_id = p.person_id
LEFT JOIN omop.concept mc ON m.measurement_concept_id = mc.concept_id
LEFT JOIN omop.concept gc ON p.gender_concept_id = gc.concept_id
WHERE m.measurement_concept_id IN ({csv(VITAL_CONCEPT_IDS)})
  AND m.value_as_number IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
""")

### Use case 2 — Mechanical ventilation: who, and for how long

`device_exposure` ⋈ `concept`. Ventilation time per exposure = end − start; reported as median / p75 hours.

In [ ]:
con.sql(f"""
SELECT dc.concept_name AS device,
       COUNT(DISTINCT de.person_id) AS n_patients,
       COUNT(*) AS n_exposures,
       round(median(date_diff('hour', de.device_exposure_start_datetime, de.device_exposure_end_datetime)), 1) AS median_hours,
       round(quantile_cont(date_diff('hour', de.device_exposure_start_datetime, de.device_exposure_end_datetime), 0.75), 1) AS p75_hours
FROM omop.device_exposure de
LEFT JOIN omop.concept dc ON de.device_concept_id = dc.concept_id
WHERE de.device_concept_id IN ({csv(VENT_DEVICE_CONCEPT_IDS)})
  AND de.device_exposure_end_datetime IS NOT NULL
GROUP BY 1
ORDER BY n_patients DESC
""")

### Use case 3 — ICU length of stay & in-stay mortality, by age band

`visit_occurrence` ⋈ `person` ⋈ `death`. *In-stay* mortality = a `death` record dated within the visit window. LOS = visit_end − visit_start.

In [ ]:
con.sql("""
WITH stays AS (
  SELECT v.visit_occurrence_id,
         date_diff('hour', v.visit_start_datetime, v.visit_end_datetime) / 24.0 AS los_days,
         date_diff('year', p.birth_datetime, v.visit_start_datetime) AS age,
         (d.death_datetime IS NOT NULL
          AND d.death_datetime BETWEEN v.visit_start_datetime AND v.visit_end_datetime) AS died_in_stay
  FROM omop.visit_occurrence v
  JOIN omop.person p ON v.person_id = p.person_id
  LEFT JOIN omop.death d ON v.person_id = d.person_id
  WHERE v.visit_end_datetime IS NOT NULL
)
SELECT CASE WHEN age < 50 THEN '<50'
            WHEN age < 65 THEN '50-64'
            WHEN age < 80 THEN '65-79'
            ELSE '80+' END AS age_band,
       COUNT(*) AS n_visits,
       round(median(los_days), 2) AS median_los_days,
       round(100.0 * AVG(CASE WHEN died_in_stay THEN 1 ELSE 0 END), 1) AS mortality_pct
FROM stays
GROUP BY 1
ORDER BY 1
""")

### Use case 4 — Vital signs: survivors vs non-survivors

Per-patient mean vital (CTE) ⋈ `death`. Mortality here = the patient has any `death` record. Grouped by vital so units stay consistent within each row.

In [ ]:
con.sql(f"""
WITH per_patient AS (
  SELECT m.measurement_concept_id, m.person_id, AVG(m.value_as_number) AS mean_vital
  FROM omop.measurement m
  WHERE m.measurement_concept_id IN ({csv(VITAL_CONCEPT_IDS)})
    AND m.value_as_number IS NOT NULL
  GROUP BY 1, 2
)
SELECT mc.concept_name AS vital,
       CASE WHEN d.person_id IS NOT NULL THEN 'died' ELSE 'survived' END AS outcome,
       COUNT(*) AS n_patients,
       round(median(pp.mean_vital), 1) AS median_mean_vital,
       round(quantile_cont(pp.mean_vital, 0.25), 1) AS p25,
       round(quantile_cont(pp.mean_vital, 0.75), 1) AS p75
FROM per_patient pp
LEFT JOIN omop.concept mc ON pp.measurement_concept_id = mc.concept_id
LEFT JOIN omop.death d ON pp.person_id = d.person_id
GROUP BY 1, 2
ORDER BY 1, 2
""")